# Notebook 03 — SCLC causal model & prediction (Plan 2)

The categorical end-to-end path: `SCLCPatient` is one class, ten data properties, all
categorical — every non-root node gets `InvertibleClassifierFCM` (E1, Gumbel-max), so this
KG exercises conditional / interventional / counterfactual prediction, evaluation,
falsification and the RDF write-back all in the discrete/exact regime (tier 1 = pgmpy exact).

Run this with the `rdfenv` kernel (registered per Plan 1). Requires notebook 01 to have
already produced `results/sclc/GES.ttl` (or another method's `.ttl`) — run
`runners/run_kg_discovery.py` first if that directory is empty.

**Runtime expectation:** fitting is seconds; `evaluate_model` (mechanism CV) is seconds;
`falsify` runs many kernel conditional-independence tests over 4279 rows and can take several
minutes — its cell is opt-in, not run by default.

In [1]:
import os, sys

def _find_root():
    p = os.getcwd()
    for _ in range(8):
        if os.path.isdir(os.path.join(p, 'causalway')) and os.path.isdir(os.path.join(p, 'algs')):
            return p
        p = os.path.dirname(p)
    raise RuntimeError('project root not found')

PROJECT_ROOT = _find_root()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import dowhy, pgmpy, sklearn, rdflib, pandas, numpy
print('dowhy', dowhy.__version__, '| pgmpy', pgmpy.__version__, '| sklearn', sklearn.__version__)
print('PROJECT_ROOT =', PROJECT_ROOT)


dowhy 0.14 | pgmpy 1.1.0 | sklearn 1.6.1
PROJECT_ROOT = /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api


## 1. Load the Plan 1 graph — from a file, and from an in-memory instance

Both are polymorphic inputs `causalway.sources.load_ocg` accepts (§3.1); loading the same
graph both ways must give identical structure.

In [2]:
from causalway.model import CausalModel
from causalway.result import OntologicalCausalGraph
from causalway.sources import load_ocg, resolve_graph

GRAPH_TTL = os.path.join(PROJECT_ROOT, 'results', 'sclc', 'GES.ttl')
KG_TTL = os.path.join(PROJECT_ROOT, 'kgs', 'ttls', 'SCLC_patients.ttl')
assert os.path.exists(GRAPH_TTL), f'{GRAPH_TTL} missing — run notebook 01 / run_kg_discovery.py first'

ocg_from_file = load_ocg(GRAPH_TTL)
ocg_from_instance = OntologicalCausalGraph.from_rdf(resolve_graph(GRAPH_TTL))

import numpy as np
assert [n.name for n in ocg_from_file.nodes] == [n.name for n in ocg_from_instance.nodes]
assert np.array_equal(ocg_from_file.adj, ocg_from_instance.adj)
print(f'{ocg_from_file.method}: {ocg_from_file.n} nodes, {int(ocg_from_file.adj.sum())} edges — file and instance loads agree ✓')
ocg_from_file.to_dataframe()


GES: 10 nodes, 15 edges — file and instance loads agree ✓


,method,cause,effect,cause_domain,effect_domain,relation,weight
0,GES,SCLCPatient.ageGroup,SCLCPatient.familyGender,SCLCPatient,SCLCPatient,ε,1.0
1,GES,SCLCPatient.biomarker,SCLCPatient.ageGroup,SCLCPatient,SCLCPatient,ε,1.0
2,GES,SCLCPatient.biomarker,SCLCPatient.episodeType,SCLCPatient,SCLCPatient,ε,1.0
3,GES,SCLCPatient.episodeType,SCLCPatient.stage,SCLCPatient,SCLCPatient,ε,1.0
4,GES,SCLCPatient.familyGender,SCLCPatient.familyCancer,SCLCPatient,SCLCPatient,ε,1.0
5,GES,SCLCPatient.gender,SCLCPatient.ageGroup,SCLCPatient,SCLCPatient,ε,1.0
6,GES,SCLCPatient.gender,SCLCPatient.biomarker,SCLCPatient,SCLCPatient,ε,1.0
7,GES,SCLCPatient.gender,SCLCPatient.familyGender,SCLCPatient,SCLCPatient,ε,1.0
8,GES,SCLCPatient.gender,SCLCPatient.smokerType,SCLCPatient,SCLCPatient,ε,1.0
9,GES,SCLCPatient.locatedIn,SCLCPatient.smokerType,SCLCPatient,SCLCPatient,ε,1.0


## 2. Fit the CausalModel

Aligns the discovered graph's nodes to the KG's materialised columns on `(domain, prop,
range)` identity (not display name), infers a dtype per column, lets `gcm.auto` pick a
mechanism class, then overrides every non-root categorical mechanism with
`InvertibleClassifierFCM` (E1) and pre-validates invertibility before fitting.

In [3]:
model = CausalModel.fit(GRAPH_TTL, KG_TTL, quality='good', random_state=0)
print('model_id:', model.model_id)
print('alignment:', model.spec.alignment)
model.mechanism_table()


Fitting causal models:   0%|          | 0/10 [00:00<?, ?it/s]

Fitting causal mechanism of node SCLCPatient.ageGroup:   0%|          | 0/10 [00:00<?, ?it/s]

Fitting causal mechanism of node SCLCPatient.ageGroup:  10%|█         | 1/10 [00:00<00:05,  1.73it/s]

Fitting causal mechanism of node SCLCPatient.biomarker:  10%|█         | 1/10 [00:00<00:05,  1.73it/s]

Fitting causal mechanism of node SCLCPatient.episodeType:  10%|█         | 1/10 [00:00<00:05,  1.73it/s]

...

Fitting causal mechanism of node SCLCPatient.smokerType:  90%|█████████ | 9/10 [00:01<00:00,  7.82it/s]

Fitting causal mechanism of node SCLCPatient.stage:  90%|█████████ | 9/10 [00:01<00:00,  7.82it/s]     

Fitting causal mechanism of node SCLCPatient.stage: 100%|██████████| 10/10 [00:01<00:00,  7.66it/s]

model_id: GES-8e3f4e312e14-8e3f4e31
alignment: {'matched': ['SCLCPatient.ageGroup', 'SCLCPatient.biomarker', 'SCLCPatient.episodeType', 'SCLCPatient.familyCancer', 'SCLCPatient.familyGender', 'SCLCPatient.gender', 'SCLCPatient.locatedIn', 'SCLCPatient.relapseStatus', 'SCLCPatient.smokerType', 'SCLCPatient.stage'], 'missing_in_kg': [], 'extra_in_kg': []}


,node,dtype,is_root,mechanism_type,invertible
0,SCLCPatient.ageGroup,categorical,False,InvertibleClassifierFCM,True
1,SCLCPatient.biomarker,categorical,False,InvertibleClassifierFCM,True
2,SCLCPatient.episodeType,categorical,False,InvertibleClassifierFCM,True
3,SCLCPatient.familyCancer,categorical,False,InvertibleClassifierFCM,True
4,SCLCPatient.familyGender,categorical,False,InvertibleClassifierFCM,True
5,SCLCPatient.gender,categorical,True,EmpiricalDistribution,True
6,SCLCPatient.locatedIn,categorical,True,EmpiricalDistribution,True
7,SCLCPatient.relapseStatus,categorical,False,InvertibleClassifierFCM,True
8,SCLCPatient.smokerType,categorical,False,InvertibleClassifierFCM,True
9,SCLCPatient.stage,categorical,False,InvertibleClassifierFCM,True


Every non-root node should be `InvertibleClassifierFCM` — the concrete payoff of E1 for an
entirely categorical KG. Root nodes keep `EmpiricalDistribution` (no mechanism to invert:
root abduction is the identity, Plan 2 §1.2).

In [4]:
table = model.mechanism_table()
non_root = table[~table['is_root']]
assert (non_root['mechanism_type'] == 'InvertibleClassifierFCM').all(), non_root
print(f"{len(non_root)} non-root nodes, all InvertibleClassifierFCM ✓ ; "
      f"{len(table) - len(non_root)} root node(s): {table[table['is_root']]['node'].tolist()}")


8 non-root nodes, all InvertibleClassifierFCM ✓ ; 2 root node(s): ['SCLCPatient.gender', 'SCLCPatient.locatedIn']


## 3. Evaluate & falsify (Plan 2 §7) — run *before* trusting any answer

`evaluate_model` wraps `gcm.evaluate_causal_model`: per-mechanism CRPS/F1, the invertibility
assumption check (independence of recovered noise from parents — reimplemented in
`causalway.evaluation` for `InvertibleClassifierFCM`'s packed Gumbel noise, see its
docstring), and the overall KL divergence between generated and observed data.

In [5]:
from causalway import evaluation

result = evaluation.evaluate_model(model, evaluate_causal_structure=False)
print(result)


Evaluating causal mechanisms...:   0%|          | 0/10 [00:00<?, ?it/s]

Evaluating causal mechanisms...: 100%|██████████| 10/10 [00:00<00:00, 6184.46it/s]

Evaluated the performance of the causal mechanisms and the invertibility assumption of the causal mechanisms and the overall average KL divergence between generated and observed distribution. The results are as follows:

==== Evaluation of Causal Mechanisms ====
The used evaluation metrics are:
- KL divergence (only for root-nodes): Evaluates the divergence between the generated and the observed distribution.
- Mean Squared Error (MSE): Evaluates the average squared differences between the observed values and the conditional expectation of the causal mechanisms.
- Normalized MSE (NMSE): The MSE normalized by the standard deviation for better comparison.
- R2 coefficient: Indicates how much variance is explained by the conditional expectations of the mechanisms. Note, however, that this can be misleading for nonlinear relationships.
- F1 score (only for categorical non-root nodes): The harmonic mean of the precision and recall indicating the goodness of the underlying classifier model.


**Held-out CV** of conditional prediction against a marginal baseline — a node the model
can't beat the marginal on makes interventional/counterfactual answers about it noise.

In [6]:
cv = evaluation.held_out_cv(model, n_splits=5, random_state=0)
cv.sort_values('macro_f1', ascending=False)


,node,metric,model,baseline,beats_baseline,macro_f1
1,SCLCPatient.biomarker,accuracy,0.705539,0.596167,True,0.667518
6,SCLCPatient.smokerType,accuracy,0.747137,0.741996,True,0.594180
4,SCLCPatient.familyGender,accuracy,0.578406,0.556672,True,0.573499
0,SCLCPatient.ageGroup,accuracy,0.848796,0.847394,True,0.481108
5,SCLCPatient.relapseStatus,accuracy,0.896237,0.896237,False,0.472640
3,SCLCPatient.familyCancer,accuracy,0.548960,0.460154,True,0.394338
7,SCLCPatient.stage,accuracy,0.669315,0.620706,True,0.281834
2,SCLCPatient.episodeType,accuracy,0.471839,0.449871,True,0.260068


`falsify` (`gcm.falsify.falsify_graph`) runs a permutation test of the graph's conditional-
independence implications — genuinely slow on 4279 rows (kernel CI tests), so this cell is
opt-in. Lower `n_permutations` trades rigor for speed; `allow_data_subset=True` (default)
lets it subsample internally.

In [7]:
RUN_FALSIFY = False  # flip to True to run it (can take several minutes)
if RUN_FALSIFY:
    falsify_result = evaluation.falsify(model, n_permutations=20, show_progress_bar=True)
    print(falsify_result)
else:
    print('Skipped — set RUN_FALSIFY = True to run gcm.falsify.falsify_graph.')


Skipped — set RUN_FALSIFY = True to run gcm.falsify.falsify_graph.


## 4. Conditional prediction — the backend ladder, as a regression test

Pick a target with at least one parent in the *discovered* graph (structure-dependent, so
found programmatically rather than hardcoded) and compare tier 0 (direct mechanism), tier 1
(`pgmpy` exact — SCLC is entirely categorical, so this tier always applies), and tier 2
(likelihood weighting). They should agree to Monte-Carlo error — this is the ladder's own
regression test (Plan 2 §4, notebook 03 step 4).

In [8]:
import networkx as nx

target = next(n for n in model.spec.columns if list(model.scm.graph.predecessors(n)))
parents = list(model.scm.graph.predecessors(target))
parent = parents[0]  # reused as the do()/counterfactual node in sections 5-6 below
# backend='mechanism' (tier 0) needs evidence covering *every* parent of target — a partial
# evidence set correctly falls through to the sampling tiers instead (condition()'s docstring).
evidence = {p: str(model.spec.data[p].mode().iloc[0]) for p in parents}
print(f'target={target}  parents={parents}  evidence={evidence}')

a0 = model.condition(target, evidence, backend='mechanism')
a1 = model.condition(target, evidence, backend='pgmpy-exact')
a2 = model.condition(target, evidence, backend='likelihood-weighting', num_samples=20_000)

import pandas as pd
pd.DataFrame({'mechanism': a0.distribution, 'pgmpy-exact': a1.distribution,
              'likelihood-weighting': a2.distribution}).round(3)


target=SCLCPatient.ageGroup  parents=['SCLCPatient.biomarker', 'SCLCPatient.gender', 'SCLCPatient.stage']  evidence={'SCLCPatient.biomarker': 'Others', 'SCLCPatient.gender': 'Male', 'SCLCPatient.stage': 'IV'}


/Users/jason/miniconda3/envs/rdfenv/lib/python3.11/site-packages/pgmpy/estimators/__init__.py:4: FutureWarning: `pgmpy.estimators.StructureScore` is deprecated and will be removed in a future release. Use `pgmpy.structure_score` instead.
  from .StructureScore import (


,mechanism,pgmpy-exact,likelihood-weighting
OLDER,0.946,0.945,0.947
YOUNGER,0.054,0.055,0.053


## 5. Interventional prediction — population-level vs. entity-level

`do(parent := level)`, marginally (drawn from the fitted model) and for one entity's own rows
(the `entity=` scoping of `intervene`, distinct from a `CounterfactualQuery`'s `aboutEntity`).
`reference=` is optional in both cases — it only adds an alt-vs-ref `.effect` contrast; a
plain call already returns `P(target | do(...))` in `.distribution`.

In [9]:
levels = sorted(model.spec.data[parent].unique())
alt_level = levels[0]
print(f'do({parent} := {alt_level!r})')

ans_pop = model.intervene({parent: alt_level}, target=target, num_samples=10_000)
print('\npopulation-level distribution:', ans_pop.distribution)

entity0 = str(model.spec.mat.entity_ids.iloc[0, 0])
ans_entity = model.intervene({parent: alt_level}, target=target, entity=entity0)
print(f'\nsame intervention, scoped to {entity0}:', ans_entity.distribution)

do(SCLCPatient.biomarker := 'ALKorEGFR')



population-level distribution: {'OLDER': 0.7828, 'YOUNGER': 0.2172}

same intervention, scoped to http://causalkg.example.org/sclc/patient/patient_0: {'OLDER': 1.0}


## 6. Counterfactual — entity-level, one abduction reused across treatments

Abduct noise once from one patient's row, then evaluate several hypothetical treatments
against the *same* noise draw — the point of `noise_data=` reuse (Plan 2 §5.3). Because the
target is categorical, the Gumbel-max abduction is stochastic, so the answer is a
distribution over `num_samples` re-abductions, not a point (§3.4).

In [10]:
for level in levels:
    ans_cf = model.counterfactual({(entity0, parent): level}, entity=entity0, target=target,
                                  num_samples=200)
    top = max(ans_cf.distribution, key=ans_cf.distribution.get)
    print(f'do({parent} := {level!r:>12}) for {entity0.rsplit("/", 1)[-1]:>12}  ->  '
          f'{target} = {top!r:>6}  (p={ans_cf.distribution[top]:.2f})   coupling={ans_cf.coupling}')


do(SCLCPatient.biomarker :=  'ALKorEGFR') for    patient_0  ->  SCLCPatient.ageGroup = 'OLDER'  (p=1.00)   coupling=gumbel-max
do(SCLCPatient.biomarker :=     'Others') for    patient_0  ->  SCLCPatient.ageGroup = 'OLDER'  (p=0.99)   coupling=gumbel-max


**Stated limitation** (Plan 2 §3.4, §10 risk #1): a categorical counterfactual is *not*
identified by the observational distribution — Gumbel-max fixes one particular coupling.
`ordinal=` routes a column through `DiscreteAdditiveNoiseModel` instead (invertible out of
the box, order-based) — a natural fit for `SCLCPatient.stage`, which has a real order
(I < II < III < IV). Refit with it and compare the same query under both couplings, to make
how much of the answer is the coupling choice (rather than the data) visible.

In [11]:
ordinal_target = 'SCLCPatient.stage'
if ordinal_target in model.spec.columns and target != ordinal_target:
    model_ordinal = CausalModel.fit(GRAPH_TTL, KG_TTL, quality='good', random_state=0,
                                    ordinal=[ordinal_target])
    print(model_ordinal.mechanism_table().query("node == @ordinal_target"))
else:
    print(f'Skipped: pick a different ordinal_target than {target!r} to compare couplings on.')


Fitting causal models:   0%|          | 0/10 [00:00<?, ?it/s]

Fitting causal mechanism of node SCLCPatient.ageGroup:   0%|          | 0/10 [00:00<?, ?it/s]

Fitting causal mechanism of node SCLCPatient.biomarker:   0%|          | 0/10 [00:00<?, ?it/s]

Fitting causal mechanism of node SCLCPatient.episodeType:   0%|          | 0/10 [00:00<?, ?it/s]

Fitting causal mechanism of node SCLCPatient.familyCancer:   0%|          | 0/10 [00:00<?, ?it/s]

...

Fitting causal mechanism of node SCLCPatient.smokerType:   0%|          | 0/10 [00:00<?, ?it/s]   

Fitting causal mechanism of node SCLCPatient.smokerType:  90%|█████████ | 9/10 [00:00<00:00, 18.74it/s]

Fitting causal mechanism of node SCLCPatient.stage:  90%|█████████ | 9/10 [00:00<00:00, 18.74it/s]     

Fitting causal mechanism of node SCLCPatient.stage: 100%|██████████| 10/10 [00:00<00:00, 15.57it/s]

                node    dtype  is_root              mechanism_type  invertible
9  SCLCPatient.stage  ordinal    False  DiscreteAdditiveNoiseModel        True


## 7. Write-back: `store_query` + a SPARQL round trip + the §6.5 reach check

Store a handful of counterfactual queries (one per patient), merge them with the *source*
KG in one graph, and run a SPARQL query that only works because node IRIs are global
(Plan 1 §3.7): from an estimate, to its target node, to the causal edges feeding it.

In [12]:
from causalway.queries import Intervention, Query, store_query, validate_query

def node_index(name: str) -> int:
    return [n.name for n in model.spec.ocg.nodes].index(name)

def node_iri(name: str):
    return model.spec.ocg.node_iri(node_index(name))

def node_prop(name: str):
    return model.spec.ocg.nodes[node_index(name)].prop

merged = resolve_graph(KG_TTL)  # start from the *source* KG — annotation mode adds to it
model.spec.ocg.to_rdf(graph=merged)  # + the causal graph itself, so §6.5 reach queries have edges to walk
sample_entities = [str(e) for e in model.spec.mat.entity_ids.iloc[:5, 0]]

for entity in sample_entities:
    q = Query(kind='counterfactual', model_id=model.model_id, target=[target],
             interventions=[Intervention(node=parent, value=alt_level, entity=entity)],
             entity=entity)
    validate_query(q, model.spec.ocg, model.spec.mat)
    ans = model.counterfactual({(entity, parent): alt_level}, entity=entity, target=target,
                               num_samples=100)
    store_query(ans, q, model, graph=merged)

print(f'{len(merged)} triples in the merged graph (source KG + {len(sample_entities)} counterfactual queries)')


47471 triples in the merged graph (source KG + 5 counterfactual queries)


In [13]:
target_prop = node_prop(target)
sparql = f"""
PREFIX cw: <http://sdm-causalway.org/>
SELECT ?patient ?predicted ?factual WHERE {{
  ?estimate a cw:CounterfactualEstimate ;
            cw:onEntity ?patient ;
            cw:onProperty <{target_prop}> ;
            cw:predictedValue ?predicted .
  ?patient <{target_prop}> ?factual .
}}
"""
rows = [(str(r.patient).rsplit('/', 1)[-1], str(r.predicted), str(r.factual))
        for r in merged.query(sparql)]
pd.DataFrame(rows, columns=['patient', 'counterfactual', 'factual'])

,patient,counterfactual,factual
0,patient_0,OLDER,OLDER
1,patient_1,OLDER,OLDER
2,patient_2,OLDER,OLDER
3,patient_3,OLDER,OLDER
4,patient_4,OLDER,OLDER


The §6.5 causal-reach check as a raw SPARQL `ASK` over the stored graph (the same check
`validate_query` runs in Python): an intervened node needs a directed path to the target.

In [14]:
from causalway.vocab import CW

def reach_ask(intervened_node_name: str, target_node_name: str) -> bool:
    iv_iri = node_iri(intervened_node_name)
    t_iri = node_iri(target_node_name)
    q = f"ASK {{ <{iv_iri}> (^cw:cause/cw:effect)+ <{t_iri}> . }}"
    return bool(merged.query(q, initNs={'cw': CW}).askAnswer)

print(f'{parent} reaches {target}:', reach_ask(parent, target), '(expected True)')

ancestors_of_target = nx.ancestors(model.scm.graph, target)
no_path_source = next((n for n in model.spec.columns
                       if n != target and n not in ancestors_of_target), None)
if no_path_source:
    print(f'{no_path_source} reaches {target}:', reach_ask(no_path_source, target), '(expected False)')
    try:
        validate_query(Query(kind='interventional', model_id=model.model_id, target=[target],
                             interventions=[Intervention(node=no_path_source, value='x')]),
                      model.spec.ocg)
        print('validate_query unexpectedly accepted an unreachable intervention')
    except ValueError as e:
        print('validate_query correctly rejected it:', e)
else:
    print('Every node reaches the target in this discovered graph — no negative example to show.')


SCLCPatient.biomarker reaches SCLCPatient.ageGroup: True (expected True)
SCLCPatient.familyCancer reaches SCLCPatient.ageGroup: False (expected False)
validate_query correctly rejected it: validate_query: no causal path from 'SCLCPatient.familyCancer' to target 'SCLCPatient.ageGroup'; this intervention cannot reach the target (§6.5 causal reach).


## 8. `replay()`

`replay` reconstructs executable `Query` objects from the stored RDF and re-runs them,
comparing against the stored `cw:predictedValue` — this is what turns the KG from a report
into an interface (the concrete deliverable for DEMANDs.md §3).

There is no SHACL shape file: the old `shapes.ttl` checked a shape nothing emits any
more and was retired. A replay is the stronger check — it asserts the stored RDF still
*means* the answer it was built from, not merely that it has the right shape.

In [15]:
from causalway.queries import replay

replay_df = replay(merged, model)
replay_df


,qid,kind,target,stored,recomputed,match
0,2105af6f7644,counterfactual,SCLCPatient.ageGroup,OLDER,OLDER,True
1,e9a515f9ce06,counterfactual,SCLCPatient.ageGroup,OLDER,OLDER,True
2,ab2b6deda99a,counterfactual,SCLCPatient.ageGroup,OLDER,OLDER,True
3,14c66481fe84,counterfactual,SCLCPatient.ageGroup,OLDER,OLDER,True
4,1696b6bf0572,counterfactual,SCLCPatient.ageGroup,OLDER,OLDER,True


In [16]:
assert replay_df['match'].all(), replay_df[~replay_df['match']]
print(f'{len(replay_df)}/{len(replay_df)} replayed queries match their stored answer ✓')


5/5 replayed queries match their stored answer ✓


## Save the model

Persists `scm.pkl` + `manifest.json`; `CausalModel.load(...)` re-resolves the causal graph
and KG from the manifest's recorded provenance, so a loaded model can answer queries
immediately (see `CausalModel.load`'s docstring).

In [17]:
out_dir = os.path.join(PROJECT_ROOT, 'results', 'models', model.model_id)
model.save(out_dir)
print('saved to', out_dir)

reloaded = CausalModel.load(out_dir)
print('reloaded ocg available:', reloaded.spec.ocg is not None,
      '| data available:', reloaded.spec.data is not None)


saved to /Users/jason/Documents/Coding Projects.nosync/2026_Claude/CausalKG_api/results/models/GES-8e3f4e312e14-8e3f4e31


reloaded ocg available: True | data available: True
